In [1]:
import pandas as pd
import os

In [14]:
model_to_check_list = ['results_shipsnet_deterministic_00',
                       'results_shipsnet_deterministic_01',
                      'results_shipsnet_deterministic_02',
                      'results_shipsnet_deterministic_03']

#model_to_check = 'results_shipsnet_deterministic_00'
all_model_combined_train_df = pd.DataFrame()
all_model_max_val_acc_df = pd.DataFrame()
all_model_combined_test_df = pd.DataFrame()

for model_to_check in model_to_check_list:
    train_results_dir = os.path.join('results_shipsnet_deterministic',model_to_check)

    search_dir = train_results_dir
    #list all .json files in the directory
    all_files = [f for f in os.listdir(search_dir)]
    csv_files = [f for f in os.listdir(search_dir) if f.endswith('.csv') and f.startswith('training_')] 

    # excluding the format, get the last 16 characters of each filename
    timestamps = [f[:-5][-15:] for f in csv_files]
    print("Timestamps found count:", len(timestamps))

    #config_files = {}
    #guide_files = {}
    model_files = {}
    #param_files = {}
    training_files = {}
    test_files = {}

    for timestamp in timestamps:
        #config_files[timestamp] = [f for f in all_files if timestamp in f and f.endswith('.json')][0]
        #guide_files[timestamp] = [f for f in all_files if timestamp in f and f.startswith('guide')][0]
        model_files[timestamp] = [f for f in all_files if timestamp in f and f.endswith('pth')][0]
        training_files[timestamp] = [f for f in all_files if timestamp in f and f.endswith('csv') and f.startswith('training_')][0]
        test_files[timestamp] = [f for f in all_files if timestamp in f and f.endswith('csv') and f.startswith('testing_')][0]
        #param_files[timestamp] = [f for f in all_files if timestamp in f and f.startswith('param')][0]

    all_dataframes = []
    for timestamp in timestamps:
        csv_file = os.path.join(search_dir, training_files[timestamp])
        df = pd.read_csv(csv_file)
        df['timestamp'] = timestamp  # add a column for the timestamp
        all_dataframes.append(df)

    all_test_dataframes = []
    for timestamp in timestamps:
        csv_file = os.path.join(search_dir, test_files[timestamp])
        df = pd.read_csv(csv_file)
        df['timestamp'] = timestamp  # add a column for the timestamp
        all_test_dataframes.append(df)

    combined_train_df = pd.concat(all_dataframes, ignore_index=True)
    combined_test_df = pd.concat(all_test_dataframes, ignore_index=True)

    max_val_acc_df = combined_train_df.groupby(['model_variant', 'act_fn'])['val_acc'].max().reset_index()
    max_val_acc_df = max_val_acc_df.sort_values(by='val_acc', ascending=False)

    all_model_combined_train_df = pd.concat([all_model_combined_train_df, combined_train_df], ignore_index=True)
    all_model_max_val_acc_df = pd.concat([all_model_max_val_acc_df, max_val_acc_df], ignore_index=True)
    all_model_combined_test_df = pd.concat([all_model_combined_test_df, combined_test_df], ignore_index=True)

Timestamps found count: 7
Timestamps found count: 7
Timestamps found count: 7
Timestamps found count: 7


In [15]:
all_model_combined_test_df

,timestamp,model_variant,act_fn,test_loss,test_acc
0,_20250807_17161,0,actRWG,0.051545,0.98375
1,_20250807_17152,0,actWG,0.086033,0.97500
2,_20250807_17150,0,relu6,0.084339,0.97125
3,_20250807_17054,0,relu,0.070419,0.97625
4,_20250807_17140,0,sigmoid,0.092593,0.97000
5,_20250807_17143,0,sin,0.103170,0.98250
6,_20250807_17125,0,tanh,0.117116,0.97125
7,_20250808_15305,1,actRWG,0.045459,0.98375
8,_20250808_15294,1,actWG,0.088900,0.97125
9,_20250808_15281,1,relu6,0.088564,0.97875


In [16]:
all_model_combined_test_df['test_acc'] = all_model_combined_test_df['test_acc'].round(4)
all_model_combined_test_df_pivot = all_model_combined_test_df.pivot(index='model_variant', columns='act_fn', values='test_acc')
all_model_combined_test_df_pivot

act_fn,actRWG,actWG,relu,relu6,sigmoid,sin,tanh
model_variant,,,,,,,
0,0.9838,0.9750,0.9762,0.9712,0.9700,0.9825,0.9712
1,0.9838,0.9712,0.9788,0.9788,0.9700,0.9712,0.9725
2,0.9825,0.9788,0.9788,0.9800,0.9638,0.9762,0.9738
3,0.9812,0.9738,0.9738,0.9775,0.9675,0.9800,0.9700


In [23]:
# average the all_model_combined_test_df_pivot across rows (model_variant )
all_model_combined_test_df_pivot_mean = all_model_combined_test_df_pivot.mean(axis=1).round(4)
# add the mean as a new column to the pivot table
all_model_combined_test_df_pivot['mean_test_acc'] = all_model_combined_test_df_pivot_mean
all_model_combined_test_df_pivot

act_fn,actRWG,actWG,relu,relu6,sigmoid,sin,tanh,mean_test_acc
model_variant,,,,,,,,
0,0.9838,0.9750,0.9762,0.9712,0.9700,0.9825,0.9712,0.9757
1,0.9838,0.9712,0.9788,0.9788,0.9700,0.9712,0.9725,0.9752
2,0.9825,0.9788,0.9788,0.9800,0.9638,0.9762,0.9738,0.9763
3,0.9812,0.9738,0.9738,0.9775,0.9675,0.9800,0.9700,0.9748
